# Kalavinka: Hindi film-song singing on Colab

Kalavinka is **SoulX-Singer fine-tuned on 10.7 hours of Hindi film songs** (IE624 project, Gokul & Rushikesh).
You type a line of Hindi lyrics; it sings them to the tune of a real song line (from songs it never trained on),
in the voice of a short clip.

**Run the cells from top to bottom.** The first run takes about 5 minutes (install + a 2.7 GB download); after that
every line takes a few seconds.

Before you start: **Runtime → Change runtime type → T4 GPU** (it also works on CPU, just much slower).

## 1. Check the GPU

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv || echo "No GPU: it will run on CPU (slow). Runtime -> Change runtime type -> T4 GPU"

## 2. Get the Kalavinka files

Choose where the zip is:
* **drive**: you put `kalavinka_colab.zip` in your Google Drive (fastest if you will use it more than once), or
* **upload**: you pick the zip from your computer (about 220 MB).

If the folder `/content/kalavinka_colab` already exists, this cell does nothing.

In [ ]:
#@title Where is kalavinka_colab.zip?
source = "drive"  #@param ["drive", "upload"]
drive_path = "/content/drive/MyDrive/kalavinka_colab.zip"  #@param {type:"string"}
import os, zipfile
if not os.path.isdir("/content/kalavinka_colab"):
    if source == "drive":
        from google.colab import drive
        drive.mount("/content/drive")
        zpath = drive_path
    else:
        from google.colab import files
        zpath = "/content/" + next(iter(files.upload()))
    zipfile.ZipFile(zpath).extractall("/content")
%cd /content/kalavinka_colab
!ls

## 3. Install the Python packages (about 1 minute)

In [ ]:
!pip install -q -r requirements.txt

## 4. Load the model

The first time this downloads SoulX-Singer's base weights (2.7 GB) from Hugging Face, then loads our fine-tune on
top. It prints `ready` when done.

In [ ]:
import sys
sys.path.insert(0, "/content/kalavinka_colab")
from IPython.display import Audio, display, Markdown
from kalavinka_app.singer import KalavinkaSinger, save_wav
from kalavinka_app.cli import print_tunes, print_voices

singer = KalavinkaSinger("/content/kalavinka_colab")

def show(wav, sr, info=None, title=""):
    """Play a take in the notebook and say what was sung."""
    if title:
        display(Markdown(f"**{title}**"))
    if info:
        for l in info["lines"]:
            display(Markdown(f"*{l['line']}* on tune {l['tune']} ({l['tune_song']}), voice {l['voice']}; "
                             f"{int(l['seen_in_training']*100)}% of its phrases occur in the training lyrics; "
                             f"made in {info['took_s']} s"))
    display(Audio(wav, rate=sr))

## 5. Hear it sing a new line

These three lines do not occur in the training lyrics.

In [ ]:
for line in ["ठंडी हवा में गीत नया", "चाँद की रोशनी में तेरा नाम", "बारिश की बूँदों ने कुछ कहा"]:
    wav, sr, info = singer.sing(line)
    show(wav, sr, info, line)

## 6. Your own lyrics

* Lyrics must be in **Devanagari** (use a Hindi keyboard, or Google Input Tools for Hinglish → हिंदी).
* Several lines: separate them with `|`; they are sung one after another.
* `tune` and `voice`: 0 = automatic. Run the next cell for the lists.
* `pitch`: **score** = the model's own melody (sounded most natural in our listening test);
  **expressive** = our two-stage system (a contour model draws a pitch curve with vibrato first).

In [ ]:
#@title Sing your lyrics
lyrics = "तेरी यादों में खोया है दिल"  #@param {type:"string"}
tune = 0  #@param {type:"integer"}
voice = 0  #@param {type:"integer"}
pitch = "score"  #@param ["score", "expressive"]
seed = 0  #@param {type:"integer"}
wav, sr, info = singer.sing(lyrics, tune or None, voice or None, pitch, seed=seed)
show(wav, sr, info)
save_wav("outputs/my_take.wav", wav, sr); print("saved outputs/my_take.wav")

In [ ]:
print_tunes(singer.bundle); print(); print_voices(singer.bundle)

## 7. Compare with the real singer

Re-sing a bundled line with its original words. **real-pitch** gives the model the real singer's own pitch curve
(the best case from our report).

In [ ]:
#@title Re-sing a tune
tune_no = 2  #@param {type:"integer"}
pitch = "score"  #@param ["score", "expressive", "real-pitch"]
out = singer.resing(tune_no, pitch)
display(Markdown(f"**{out['song']}**: {out['lyric']}"))
show(*out["model"], title="Kalavinka")
show(*out["real"], title="Real singer")

## 8. The model's own pitch vs the two-stage (expressive) pitch, side by side

In [ ]:
line = "ठंडी हवा में गीत नया"
for pitch in ["score", "expressive"]:
    wav, sr, info = singer.sing(line, mode=pitch)
    show(wav, sr, info, pitch)

## 9. Base SoulX-Singer vs Kalavinka, on new lyrics and on alankars

`model="base"` is the released SoulX-Singer with Hindi phones mapped to English ones (our zero-shot baseline).
Alankars are sargam exercises: every note is a known swara, so we can check note by note how close each model sings
to the written pitch (within 50 cents = the right swara, within 25 cents = in tune). Notation: `Sa Re Ga | Re Ga ma`,
komal `re(k)`, tivra `Ma(t)`, upper `Sa'`, lower `.Ni`, long `Pa:2`, hold `-`, rest `_`.
To let ChatGPT write a score: `print(LLM_PROMPT)`, paste it there, describe the alankar, and pass the JSON answer to
`from_json`.

In [ ]:
from kalavinka_app.sargam import ALANKARS, LLM_PROMPT, SargamSinger, alankar_lines, from_json, parse_sargam, to_text
ss = SargamSinger(singer)
line = "ठंडी हवा में गीत नया"
for model in ["kalavinka", "base"]:
    wav, sr, info = singer.sing(line, model=model)
    print(model); display(Audio(wav, rate=sr))

In [ ]:
#@title An alankar, both models
alankar = 2  #@param {type:"integer"}
my_swaras = ""  #@param {type:"string"}
voice = 1  #@param {type:"integer"}
lines = parse_sargam(my_swaras) if my_swaras.strip() else alankar_lines(alankar)
print(to_text(lines))
for model in ["kalavinka", "base"]:
    wav, sr, info = ss.sing(lines, model=model, voice=voice)
    p = info["pitch"]
    print(f"{model}: {p['within_50c']*100:.0f}% notes on the swara, {p['within_25c']*100:.0f}% in tune, "
          f"mean error {p['mean_abs_cents']} cents")
    display(Audio(wav, rate=sr))

## 9b. Compose your own melody (pitch, timing, ornaments)
A composition is a JSON file: swaras, lengths, lyrics and ornaments (meend, gamak, kan, murki, andolan, vibrato ...). The web page's **Compose** tab (section 10) edits it on the pitch curve with the mouse. Here: the built-in Yaman example, sung by both models, with a check of how closely each followed the curve.

In [ ]:
!python -m kalavinka_app compose --example --save example.json
!python -m kalavinka_app compose example.json --model both
!python -m kalavinka_app compose-prompt | head -30   # give this to ChatGPT to write a composition

## 10. (Optional) The web page

Opens the same thing as a small web app with a public link (valid for 72 hours). Stop the cell to close it.

In [ ]:
from kalavinka_app.app import launch
launch(singer.bundle, share=True)

---
Licences and rules for the audio clips: see `NOTICE.md` (the tune and voice clips are from commercial songs and are
for this course project only).